# Lesson 01 — Load raw AdventureWorks CSVs into the Bronze layer

The raw CSV files in `/Volumes/adventure_works/default/raw_data` have **no header rows**.
Instead of guessing column names, we parse `instawdb.sql` (the original SQL Server schema script) to extract:

* the **exact column names** of each table (from the `CREATE TABLE` statements)
* the **field delimiter** of the CSV files (from the `BULK INSERT` statements)

Each CSV is then loaded as a Delta table (overwrite mode) into `adventure_works.bronze` with snake_case names:
`product`, `sales_order_detail`, `sales_territory`, `purchase_order_detail`, `purchase_order_header`, `vendor`.

In [0]:
import re

SQL_SCRIPT_PATH = "/Volumes/adventure_works/default/raw_data/instawdb.sql"
RAW_DATA_DIR = "/Volumes/adventure_works/default/raw_data"
BRONZE_SCHEMA = "adventure_works.bronze"

# SQL Server table name (as used in instawdb.sql) -> bronze Delta table name
TABLES = {
    "Product": "product",
    "SalesOrderDetail": "sales_order_detail",
    "SalesTerritory": "sales_territory",
    "PurchaseOrderDetail": "purchase_order_detail",
    "PurchaseOrderHeader": "purchase_order_header",
    "Vendor": "vendor",
}

sql_script = open(SQL_SCRIPT_PATH, encoding="utf-8", errors="replace").read()


def parse_columns(table_name):
    """Extract the exact column names of a table from its CREATE TABLE statement."""
    create_block = re.search(
        r"CREATE TABLE \[\w+\]\.\[" + table_name + r"\]\((.*?)\r?\n\)\s*ON",
        sql_script,
        re.S,
    ).group(1)
    # Column lines start with [ColumnName]; CONSTRAINT lines start with a keyword
    # and therefore never match. This also picks up computed columns (e.g. LineTotal)
    # and rowguid, because the CSV files contain values for them too.
    return re.findall(r"^\s*\[([A-Za-z0-9_]+)\]", create_block, re.M)


def parse_delimiter(table_name):
    """Extract the CSV field delimiter from the BULK INSERT statement of a table."""
    raw = re.search(
        r"BULK INSERT \[\w+\]\.\[" + table_name + r"\].*?FIELDTERMINATOR\s*=\s*'([^']+)'",
        sql_script,
        re.S,
    ).group(1)
    # The SQL script encodes the tab character as the literal two characters \t
    return "\t" if raw == "\\t" else raw


table_specs = {}
for sql_name, bronze_name in TABLES.items():
    table_specs[sql_name] = {
        "bronze_name": bronze_name,
        "columns": parse_columns(sql_name),
        "delimiter": parse_delimiter(sql_name),
    }
    spec = table_specs[sql_name]
    print(f"{sql_name}: {len(spec['columns'])} columns, delimiter={spec['delimiter']!r}")

In [0]:
# Make sure the bronze schema exists, then load every CSV as a Delta table
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")

for sql_name, spec in table_specs.items():
    csv_path = f"{RAW_DATA_DIR}/{sql_name}.csv"

    # header=False: the raw CSV files have no header row, so Spark names columns _c0, _c1, ...
    df = (
        spark.read
        .option("header", False)
        .option("sep", spec["delimiter"])
        .csv(csv_path)
    )

    # Sanity check: the number of CSV columns must match the SQL script definition
    assert len(df.columns) == len(spec["columns"]), (
        f"{sql_name}: CSV has {len(df.columns)} columns "
        f"but instawdb.sql defines {len(spec['columns'])}"
    )

    # Rename _c0, _c1, ... to the real column names extracted from the SQL script
    for spark_col, real_col in zip(df.columns, spec["columns"]):
        df = df.withColumnRenamed(spark_col, real_col)

    row_count = df.count()
    (
        df.write
        .format("delta")
        .mode("overwrite")
        # overwriteSchema: bronze keeps raw STRING values, so the table schema
        # is replaced instead of merged with an older typed version of the table
        .option("overwriteSchema", "true")
        .saveAsTable(f"{BRONZE_SCHEMA}.{spec['bronze_name']}")
    )
    print(f"Loaded {BRONZE_SCHEMA}.{spec['bronze_name']} ({row_count} rows)")

In [0]:
# Verify the load: row counts for all 6 bronze tables
row_counts = [
    (f"{BRONZE_SCHEMA}.{spec['bronze_name']}", spark.table(f"{BRONZE_SCHEMA}.{spec['bronze_name']}").count())
    for spec in table_specs.values()
]
display(spark.createDataFrame(row_counts, ["bronze_table", "row_count"]))